# Game Controllers

A keyboard key is either up or down. A **game controller** can say much more: how far a stick is pushed and in which direction, how hard a trigger is squeezed, and which of a dozen buttons are held. It can even shake in your hands. In this notebook you'll connect a controller to Python, read everything it can tell you, and use it to steer a 3D scene and a Pygame game.

You need an Xbox, PlayStation, or Switch Pro controller, or another one that follows the same layout.

## Before you start

**Pair the controller with your computer first**, the same way you would for any game: plug it in with a USB cable, or connect it in your computer's Bluetooth settings. Codetto reads controllers your computer already knows about. It does not do the pairing.

Then run the cell below. Browsers keep controllers hidden from web pages until a button is pressed, so `connect()` prints a message and waits. **Press any button on the controller** and the cell finishes.

In [ ]:
from codetto import devices

pad = devices.Gamepad.connect()

print('Connected:', pad.name)

`pad` now stands for your controller, and every other cell in this notebook uses it. If you switch the controller off or reload the page, run the connect cell again.

Two options are useful later on:

- `devices.Gamepad.connect(index=1)` waits for a **second** controller, for two-player games.
- `devices.Gamepad.connect(timeout=10)` gives up with an error after ten seconds instead of waiting until you press Stop.

# Reading the State

A controller doesn't send messages like "the stick moved". Instead you ask it, whenever you like, "what are you doing *right now*?" The answer is a **snapshot** of every stick, trigger, and button at that moment, and you get one with `pad.read()`.

| In the snapshot | What it holds |
|---|---|
| `left_stick`, `right_stick` | An `(x, y)` pair, each from `-1` to `1` |
| `left_trigger`, `right_trigger` | How far the trigger is pulled, from `0` to `1` |
| `dpad` | An `(x, y)` pair, each `-1`, `0`, or `1` |
| `is_pressed(button)` | `True` while that button is held down |

Hold the left stick in any direction, squeeze a trigger, and run the cell while you are still holding them. Run it again with the controller at rest to see the difference.

In [ ]:
state = pad.read()

print('Left stick:', state.left_stick)
print('Right stick:', state.right_stick)
print('Left trigger:', state.left_trigger)
print('Right trigger:', state.right_trigger)
print('D-pad:', state.dpad)
print('A held:', state.is_pressed(devices.Button.A))

# Sticks and Triggers

One snapshot shows one moment. To follow the controller as it moves, read it again and again in a loop, many times a second.

A stick gives two numbers. `x` is `-1` at the far left and `1` at the far right. `y` is `-1` at the **top** and `1` at the bottom, because `y` grows downward, the same as on a canvas. Push a stick halfway and you get about `0.5`, so a stick controls speed as well as direction.

A stick at rest never sits at exactly zero. To stop things drifting, small movements are read as `0`. That area is the **dead zone**, and you can change its size with `pad.dead_zone` (it starts at `0.1`).

The loop below moves a dot with the **left stick**. The **right trigger** makes it grow. Press **B** to finish, or use the Stop button.

In [ ]:
from codetto import graphics2d
import math
import time

canvas = graphics2d.canvas()
ctx = canvas.get_context('2d')
width, height = canvas.get_width(), canvas.get_height()
x, y = width / 2, height / 2

while True:
  state = pad.read()
  if state.is_pressed(devices.Button.B) or not state.connected:
    break

  stick_x, stick_y = state.left_stick
  x = min(max(x + stick_x * 8, 0), width)
  y = min(max(y + stick_y * 8, 0), height)
  radius = 15 + 40 * state.right_trigger

  canvas.clear()
  ctx.fill_style = '#e94560'
  ctx.begin_path()
  ctx.arc(x, y, radius, 0, 2 * math.pi)
  ctx.fill()
  time.sleep(1 / 30)

print('Finished.')

# Buttons

Buttons have names in `devices.Button`, following the Xbox layout:

| Names | Buttons |
|---|---|
| `A`, `B`, `X`, `Y` | The four face buttons |
| `LB`, `RB` | The bumpers, above the triggers |
| `LT`, `RT` | The triggers, used as buttons |
| `VIEW`, `MENU` | The two small buttons in the middle |
| `LEFT_STICK`, `RIGHT_STICK` | Clicking a stick in |
| `DPAD_UP`, `DPAD_DOWN`, `DPAD_LEFT`, `DPAD_RIGHT` | The four D-pad directions |

The names go by *position*, not by the letter printed on the button. `Button.A` is always the bottom face button. That is ✕ on a PlayStation controller, and the button labelled B on a Switch Pro controller.

There are two different questions you can ask about a button:

- `state.is_pressed(button)` asks "is it **held down** right now?" It stays `True` for as long as the button is held. Use it for things that continue, like accelerating.
- `state.was_pressed(button)` asks "has it been **pressed since my last read**?" It is `True` once for each press, even for a very quick tap. Use it for things that should happen once, like jumping. `state.was_released(button)` is the same for letting go.

Run the cell and press some buttons. Press **MENU** to finish.

In [ ]:
import time

names = {
  devices.Button.A: 'A', devices.Button.B: 'B',
  devices.Button.X: 'X', devices.Button.Y: 'Y',
  devices.Button.LB: 'LB', devices.Button.RB: 'RB',
}

pad.read()  # forget any presses from before this cell
print('Press A, B, X, Y, LB, or RB. Press MENU to finish.')

while True:
  state = pad.read()
  if state.was_pressed(devices.Button.MENU) or not state.connected:
    break

  for button, name in names.items():
    if state.was_pressed(button):
      print(name, 'pressed')
    if state.was_released(button):
      print(name, 'released')

  time.sleep(0.05)

print('Finished.')

# Rumble

A controller can talk back. Most have two small motors inside that make it shake: a **strong** one for heavy thumps and a **weak** one for a light buzz.

```python
pad.rumble(0.3, strong=1.0, weak=0.5)
```

The first number is how long to shake, in seconds. `strong` and `weak` set each motor from `0` (off) to `1` (full power).

Hold the controller and drag the sliders. `rumble` returns `False` when the browser or controller can't do it, which is the case in Firefox, so your code can carry on either way.

In [ ]:
STRONG = 0.8 #@param {type:"slider", min:0, max:1, step:0.1}
WEAK = 0.4 #@param {type:"slider", min:0, max:1, step:0.1}
SECONDS = 0.3 #@param {type:"slider", min:0.1, max:1, step:0.1}

worked = pad.rumble(SECONDS, strong=STRONG, weak=WEAK)

if worked:
  print(f'Rumbled for {SECONDS} seconds.')
else:
  print("This controller or browser can't rumble.")

# A Controller in a 3D Scene

A 3D scene already has the two things a controller needs. If scenes are new to you, [[80FE9F9A-F226-4BF4-AC1B-D5A4B4F7647E|Interactive 3D]] covers the ideas this cell builds on.

- **Buttons are events.** `scene.on_button(pad, button, handler)` calls your function once for each press, just like `scene.on_key` does for the keyboard.
- **Sticks are read every frame.** Call `pad.read()` inside the `on_frame` function and use the stick to decide how far to move. Multiplying by `dt` keeps the speed the same on every computer.

One detail matters for steering. Pushing the stick *up* gives a negative `y`, but "forward" in the scene is a positive `z`. So the code subtracts: `z - stick_y`.

Run the cell, then roll the ball with the **left stick**. **A** jumps, with a rumble, and **B** changes the color. Press Stop (■) when you are done.

In [ ]:
from codetto import graphics3d

scene = graphics3d.Scene()
scene.set_sky(graphics3d.Sky.PURE_SKY)
ground = scene.set_ground(length=20, width=20)
ground.set_material(graphics3d.Material.Grass.Bright)
ground.set_tiling(10)

ball = graphics3d.Shapes.Sphere(diameter=1, segments=24)
ball.set_position(0, 0.5, 0)
ball.set_color('#e94560')
scene.add(ball)
scene.camera.follow(ball, distance=14)

colors = ['#e94560', '#f5a623', '#4488ff', '#44cc88']
color = 0
rise = 0  # upward speed, used for jumping

def jump():
  global rise
  if ball.get_position()[1] <= 0.5:
    rise = 6
    pad.rumble(0.15, strong=0.6, weak=0.3)

def next_color():
  global color
  color = (color + 1) % len(colors)
  ball.set_color(colors[color])

scene.on_button(pad, devices.Button.A, jump)
scene.on_button(pad, devices.Button.B, next_color)

@scene.on_frame
def move(dt):
  global rise
  stick_x, stick_y = pad.read().left_stick
  x, y, z = ball.get_position()
  x = max(-9, min(9, x + stick_x * 6 * dt))
  z = max(-9, min(9, z - stick_y * 6 * dt))
  rise -= 15 * dt  # gravity
  y = max(0.5, y + rise * dt)
  ball.set_position(x, y, z)

scene.run()

# A Controller in Pygame

[[699987CF-F87D-4E8C-9F64-C8DCBB6DC0FF|Pygame]] has its own way of reading controllers, called `pygame.joystick`. It is the same code you would write for Pygame on any computer, so tutorials you find elsewhere work here too. You don't use `pad` in this section.

Pygame fits a controller into its game loop:

- When a controller appears, you get a `JOYDEVICEADDED` **event**. Create a `Joystick` from it and keep it.
- A press arrives as a `JOYBUTTONDOWN` event, once per press, like `KEYDOWN` for the keyboard.
- Sticks are read every frame with `joystick.get_axis(number)`, like `pygame.key.get_pressed()`.

Pygame uses numbers instead of names:

| Number | As an axis | As a button |
|---|---|---|
| `0` | Left stick x | A |
| `1` | Left stick y | B |
| `2` | Left trigger | X |
| `3` | Right stick x | Y |
| `4` | Right stick y | LB |
| `5` | Right trigger | RB |

Sticks run from `-1` to `1`, as before. Triggers are different here: they rest at `-1` and reach `1` when fully pulled.

Run the cell and move the square with the **left stick**. **A** changes its color and rumbles, and **B** ends the game. If nothing happens, press a button on the controller so the browser shows it to Pygame.

In [ ]:
import pygame

pygame.init()
WIDTH, HEIGHT = 400, 300
screen = pygame.display.set_mode((WIDTH, HEIGHT))
clock = pygame.time.Clock()

player = pygame.Rect(180, 130, 40, 40)
colors = ["tomato", "gold", "skyblue", "limegreen"]
color = 0
joystick = None

running = True
while running:
  for event in pygame.event.get():
    if event.type == pygame.JOYDEVICEADDED:
      joystick = pygame.joystick.Joystick(event.device_index)
      print('Connected:', joystick.get_name())
    if event.type == pygame.JOYBUTTONDOWN:
      if event.button == 0:  # A
        color = (color + 1) % len(colors)
        joystick.rumble(0.6, 0.3, 150)
      if event.button == 1:  # B
        running = False

  if joystick:
    player.x += round(joystick.get_axis(0) * 5)
    player.y += round(joystick.get_axis(1) * 5)
    player.clamp_ip(screen.get_rect())

  screen.fill((25, 30, 60))
  pygame.draw.rect(screen, colors[color], player)
  pygame.display.flip()
  clock.tick(60)

print('Finished.')

# Where to Go Next

You have the full set now: connecting, snapshots, sticks and triggers, held buttons and taps, rumble, and two ways to put a controller into a game.

Some things to try:

- Add controller steering to the coin game at the end of [[80FE9F9A-F226-4BF4-AC1B-D5A4B4F7647E|Interactive 3D]], or to the one in [[699987CF-F87D-4E8C-9F64-C8DCBB6DC0FF|Pygame]].
- Use the right stick to move the camera, or the right trigger as an accelerator.
- Rumble when the player hits a wall, with a stronger shake for a faster hit.
- Connect a second controller with `connect(index=1)` and make a two-player version of [[F7A273F5-5F3B-474D-96F9-F4D631C97464|Pong]].

# Check Your Understanding